# 03 Data Cleaning E-Commerce

Notebook ini fokus hanya pada folder `data/raw/Indonesian_Ecommerce_sales`.

Tujuan:
- Baca 6 file e-commerce bulanan.
- Bedah struktur, missing value, status pesanan, amount, dan outlier.
- Bersihkan data sampai rapi.
- Gabungkan semua bulan menjadi 1 dataset utuh.
- Simpan 1 file clean ke `data/clean/ecommerce_sales_clean.csv`.

Output utama:
- `data_bersih`: dataset e-commerce final di memory.
- `ecommerce_sales_clean.csv`: file clean final di `data/clean`.


## 1. Import library

Pakai library sederhana saja. `pandas` untuk data, `numpy` untuk angka, `matplotlib` untuk visual singkat. Modul `zipfile` dan `xml` dipakai agar Excel tetap bisa dibaca meski `openpyxl` tidak ada.


In [17]:
from pathlib import Path
from zipfile import ZipFile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')


## 2. Konfigurasi path

Raw data diambil dari folder `data/raw/Indonesian_Ecommerce_sales`. Hasil akhir disimpan ke `data/clean` sebagai satu dataset clean.


In [18]:
root_project = Path.cwd().resolve()
if root_project.name == 'notebooks':
    root_project = root_project.parent

folder_raw = root_project / 'data' / 'raw' / 'Indonesian_Ecommerce_sales'
folder_clean = root_project / 'data' / 'clean'
folder_clean.mkdir(parents=True, exist_ok=True)

print('Root project :', root_project)
print('Folder raw   :', folder_raw)
print('Folder clean :', folder_clean)

if not folder_raw.exists():
    raise FileNotFoundError(f'Folder raw tidak ditemukan: {folder_raw}')


Root project : /home/umaygans/05_nayyara_submission_1/nayyara_capstone
Folder raw   : /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/raw/Indonesian_Ecommerce_sales
Folder clean : /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/clean


## 3. Daftar file e-commerce

Ada 6 file bulanan. Dua file tidak punya kolom `Waktu Pesanan Dibuat`, jadi nanti tanggalnya dibuat dari periode bulan dataset.


In [19]:
katalog_file = [
    {'dataset_id': 'ecommerce_2024_01_january', 'nama_dataset': 'January 2024', 'periode': '2024-01', 'nama_file': '01_JanuarySales2024_clean.xlsx'},
    {'dataset_id': 'ecommerce_2024_06_june', 'nama_dataset': 'June 2024', 'periode': '2024-06', 'nama_file': '02_JuneSales2024_clean.xlsx'},
    {'dataset_id': 'ecommerce_2024_12_december', 'nama_dataset': 'December 2024', 'periode': '2024-12', 'nama_file': '03_DecemberSales2024_clean.xlsx'},
    {'dataset_id': 'ecommerce_2025_02_february', 'nama_dataset': 'February 2025', 'periode': '2025-02', 'nama_file': '04_FebruarySales2025_clean.xlsx'},
    {'dataset_id': 'ecommerce_2025_07_july', 'nama_dataset': 'July 2025', 'periode': '2025-07', 'nama_file': '05_JulySales2025_clean.xlsx'},
    {'dataset_id': 'ecommerce_2025_11_november', 'nama_dataset': 'November 2025', 'periode': '2025-11', 'nama_file': '06_NovemberSales2025_clean.xlsx'},
]

daftar_file = pd.DataFrame(katalog_file)
daftar_file['path'] = daftar_file['nama_file'].map(lambda nama: folder_raw / nama)
daftar_file['file_ada'] = daftar_file['path'].map(lambda path: path.exists())

display(daftar_file[['dataset_id', 'nama_dataset', 'periode', 'nama_file', 'file_ada']])

if not daftar_file['file_ada'].all():
    file_hilang = daftar_file.loc[~daftar_file['file_ada'], 'nama_file'].tolist()
    raise FileNotFoundError(f'File belum lengkap: {file_hilang}')


,dataset_id,nama_dataset,periode,nama_file,file_ada
0,ecommerce_2024_01_january,January 2024,2024-01,01_JanuarySales2024_clean.xlsx,True
1,ecommerce_2024_06_june,June 2024,2024-06,02_JuneSales2024_clean.xlsx,True
2,ecommerce_2024_12_december,December 2024,2024-12,03_DecemberSales2024_clean.xlsx,True
3,ecommerce_2025_02_february,February 2025,2025-02,04_FebruarySales2025_clean.xlsx,True
4,ecommerce_2025_07_july,July 2025,2025-07,05_JulySales2025_clean.xlsx,True
5,ecommerce_2025_11_november,November 2025,2025-11,06_NovemberSales2025_clean.xlsx,True


## 4. Helper baca Excel

Logic singkat:
- Coba baca Excel pakai `pd.read_excel`.
- Jika gagal karena `openpyxl` tidak ada, baca isi `.xlsx` langsung dari XML.


In [20]:
def buat_nama_kolom_unik(kolom):
    hitung = {}
    hasil = []
    for nomor, nama in enumerate(kolom):
        nama_bersih = str(nama).strip() if pd.notna(nama) and str(nama).strip() else f'kolom_{nomor + 1}'
        if nama_bersih in hitung:
            hitung[nama_bersih] += 1
            nama_bersih = f'{nama_bersih}_{hitung[nama_bersih]}'
        else:
            hitung[nama_bersih] = 0
        hasil.append(nama_bersih)
    return hasil


def nomor_kolom_excel(referensi_cell):
    huruf = ''.join(karakter for karakter in str(referensi_cell) if karakter.isalpha()) or 'A'
    nomor = 0
    for karakter in huruf.upper():
        nomor = nomor * 26 + ord(karakter) - ord('A') + 1
    return nomor - 1


def baca_excel_tanpa_engine(path_file):
    namespace = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

    with ZipFile(path_file) as workbook:
        shared_strings = []
        if 'xl/sharedStrings.xml' in workbook.namelist():
            root = ET.fromstring(workbook.read('xl/sharedStrings.xml'))
            for item in root.findall('main:si', namespace):
                teks = [node.text or '' for node in item.findall('.//main:t', namespace)]
                shared_strings.append(''.join(teks))

        nama_sheet = sorted(nama for nama in workbook.namelist() if nama.startswith('xl/worksheets/sheet'))[0]
        root = ET.fromstring(workbook.read(nama_sheet))
        semua_baris = []

        for baris in root.findall('.//main:row', namespace):
            nilai_baris = []
            for cell in baris.findall('main:c', namespace):
                indeks = nomor_kolom_excel(cell.attrib.get('r', 'A1'))
                while len(nilai_baris) <= indeks:
                    nilai_baris.append(np.nan)

                tipe = cell.attrib.get('t')
                node_value = cell.find('main:v', namespace)
                if tipe == 'inlineStr':
                    teks = [node.text or '' for node in cell.findall('.//main:t', namespace)]
                    nilai = ''.join(teks) if teks else np.nan
                elif node_value is None:
                    nilai = np.nan
                elif tipe == 's':
                    nilai = shared_strings[int(node_value.text)]
                else:
                    nilai = node_value.text
                nilai_baris[indeks] = nilai
            semua_baris.append(nilai_baris)

    jumlah_kolom = max(len(baris) for baris in semua_baris)
    semua_baris = [baris + [np.nan] * (jumlah_kolom - len(baris)) for baris in semua_baris]
    return pd.DataFrame(semua_baris[1:], columns=buat_nama_kolom_unik(semua_baris[0])).replace(r'^\s*$', np.nan, regex=True)


def baca_excel(path_file):
    try:
        return pd.read_excel(path_file).replace(r'^\s*$', np.nan, regex=True)
    except ImportError:
        return baca_excel_tanpa_engine(path_file)


## 5. Load semua file raw

Setiap bulan diberi metadata `dataset_id`, `nama_dataset`, dan `periode`, lalu digabung ke `data_mentah`.


In [21]:
list_data = []
ringkasan_raw = []

for info in katalog_file:
    path_file = folder_raw / info['nama_file']
    data_bulan = baca_excel(path_file)
    data_bulan['dataset_id'] = info['dataset_id']
    data_bulan['nama_dataset'] = info['nama_dataset']
    data_bulan['periode'] = info['periode']
    data_bulan['sumber_file'] = info['nama_file']

    list_data.append(data_bulan)
    ringkasan_raw.append({
        'dataset_id': info['dataset_id'],
        'nama_dataset': info['nama_dataset'],
        'baris_raw': len(data_bulan),
        'kolom_raw': data_bulan.shape[1],
        'punya_tanggal': 'Waktu Pesanan Dibuat' in data_bulan.columns,
    })

data_mentah = pd.concat(list_data, ignore_index=True)
ringkasan_raw = pd.DataFrame(ringkasan_raw)

print('Shape data mentah:', data_mentah.shape)
display(ringkasan_raw)
display(data_mentah.head())


Shape data mentah: (5196, 22)


,dataset_id,nama_dataset,baris_raw,kolom_raw,punya_tanggal
0,ecommerce_2024_01_january,January 2024,431,22,True
1,ecommerce_2024_06_june,June 2024,697,22,True
2,ecommerce_2024_12_december,December 2024,1214,21,False
3,ecommerce_2025_02_february,February 2025,957,22,True
4,ecommerce_2025_07_july,July 2025,766,21,False
5,ecommerce_2025_11_november,November 2025,1131,22,True


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,dataset_id,nama_dataset,periode,sumber_file
0,ORD_0006556,2,50,0,0,Aksesoris ID Card,1,Selesai,NaN,Hemat Kargo-SPX Hemat,Saldo ShopeePay,KAB. BEKASI,JAWA BARAT,0,8000,10000,8000,2024-01-01 00:05,ecommerce_2024_01_january,January 2024,2024-01,01_JanuarySales2024_clean.xlsx
1,ORD_0006557,2,1000,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA BANDUNG,JAWA BARAT,0,10000,35663,10000,2024-01-01 00:07,ecommerce_2024_01_january,January 2024,2024-01,01_JanuarySales2024_clean.xlsx
2,ORD_0006558,1,600,0,0,Plastik / Wadah Plastik,1,Selesai,NaN,Reguler (Cashless)-SPX Standard,COD (Bayar di Tempat),KAB. BOGOR,JAWA BARAT,0,10000,23187,10000,2024-01-01 00:07,ecommerce_2024_01_january,January 2024,2024-01,01_JanuarySales2024_clean.xlsx
3,ORD_0006559,1,700,0,0,Rak / Rak Serbaguna,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA DEPOK,JAWA BARAT,0,8000,37400,8000,2024-01-01 00:12,ecommerce_2024_01_january,January 2024,2024-01,01_JanuarySales2024_clean.xlsx
4,ORD_0006560,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,Online Payment,KAB. BEKASI,JAWA BARAT,0,8000,21800,8000,2024-01-01 00:36,ecommerce_2024_01_january,January 2024,2024-01,01_JanuarySales2024_clean.xlsx


## 6. Bedah kualitas data awal

Cek ini menjawab: jumlah status pesanan, missing value, duplikasi order, amount 0, dan outlier kasar sebelum cleaning.


In [22]:
missing_awal = (
    data_mentah.isna().sum()
    .reset_index(name='jumlah_missing')
    .rename(columns={'index': 'kolom'})
)
missing_awal['persen_missing'] = (missing_awal['jumlah_missing'] / len(data_mentah) * 100).round(2)

status_awal = data_mentah['Status Pesanan'].value_counts(dropna=False).reset_index(name='jumlah_baris')
produk_awal = data_mentah['product_categories'].value_counts(dropna=False).head(15).reset_index(name='jumlah_baris')

amount_awal = pd.to_numeric(data_mentah['Total Pembayaran'], errors='coerce')
q1 = amount_awal.quantile(0.25)
q3 = amount_awal.quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print('Total baris raw:', len(data_mentah))
print('Order ID duplicate:', int(data_mentah['order_id'].duplicated().sum()))
print('Amount 0 atau negatif:', int((amount_awal <= 0).sum()))
print('Outlier amount raw:', int(((amount_awal < batas_bawah) | (amount_awal > batas_atas)).sum()))

display(missing_awal.sort_values('persen_missing', ascending=False).head(10))
display(status_awal)
display(produk_awal)


Total baris raw: 5196
Order ID duplicate: 0
Amount 0 atau negatif: 651
Outlier amount raw: 571


,kolom,jumlah_missing,persen_missing
8,Alasan Pembatalan,4563,87.82
17,Waktu Pesanan Dibuat,1980,38.11
2,total_weight_gr,0,0.00
3,total_returned_qty,0,0.00
0,order_id,0,0.00
1,total_qty,0,0.00
5,product_categories,0,0.00
4,Total Diskon,0,0.00
7,Status Pesanan,0,0.00
6,num_product_categories,0,0.00


,Status Pesanan,jumlah_baris
0,Selesai,4313
1,Batal,633
2,Sedang Dikirim,59
3,"Pesanan diterima, namun Pembeli masih dapat me...",38
4,"Pesanan diterima, namun Pembeli masih dapat me...",34
5,"Pesanan diterima, namun Pembeli masih dapat me...",34
6,Telah Dikirim,30
7,"Pesanan diterima, namun Pembeli masih dapat me...",25
8,"Pesanan diterima, namun Pembeli masih dapat me...",22
9,"Pesanan diterima, namun Pembeli masih dapat me...",6


,product_categories,jumlah_baris
0,Celengan,1309
1,Mangkok Sambal / Saus,1150
2,Aksesoris Pintu,745
3,Nampan / Tray,283
4,Seal / Baut / Roof,204
5,Other,124
6,Lunch Box / Rantang,123
7,Rak / Rak Serbaguna,122
8,Keranjang,111
9,Tempat Nasi,110


## 8. Helper cleaning sederhana

Logic helper:
- `ambil_kolom`: aman jika kolom tidak ada.
- `ubah_angka`: ubah teks angka ke numeric.
- `ubah_tanggal`: parse tanggal campuran.
- `tanggal_dari_periode`: fallback untuk bulan tanpa kolom tanggal.


In [23]:
def ambil_kolom(data, nama_kolom, default=pd.NA):
    if nama_kolom in data.columns:
        return data[nama_kolom]
    return pd.Series(default, index=data.index)


def bersihkan_teks(series, default='unknown'):
    hasil = series.astype('string').str.strip()
    hasil = hasil.replace('', pd.NA).fillna(default)
    return hasil


def ubah_angka(series):
    angka = series.astype('string').str.replace(r'[^0-9.\-]', '', regex=True)
    return pd.to_numeric(angka, errors='coerce').astype(float)


def ubah_tanggal(series):
    return pd.to_datetime(series, errors='coerce', format='mixed')


def tanggal_dari_periode(series_periode):
    return pd.to_datetime(series_periode.astype('string') + '-01', errors='coerce')


def hitung_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    bawah = q1 - 1.5 * iqr
    atas = q3 + 1.5 * iqr
    return q1, q3, iqr, bawah, atas


## 9. Cleaning utama

Logic cleaning:
1. Pilih dan rename kolom penting supaya rapi.
2. Buat `tanggal_pesanan`. Jika tanggal asli kosong/tidak ada, pakai tanggal awal bulan dari `periode`.
3. Ubah kolom angka ke numeric.
4. Hapus order `Batal`, karena bukan transaksi penjualan berhasil.
5. Hapus `total_pembayaran <= 0`.
6. Hapus duplicate berdasarkan `order_id`.
7. Tandai outlier memakai IQR, lalu buat versi capped tanpa menghapus nilai original.


In [28]:
data_bersih = pd.DataFrame()

data_bersih['order_id'] = bersihkan_teks(ambil_kolom(data_mentah, 'order_id'))
data_bersih['dataset_id'] = data_mentah['dataset_id']
data_bersih['nama_dataset'] = data_mentah['nama_dataset']
data_bersih['periode'] = data_mentah['periode']

tanggal_asli = ubah_tanggal(ambil_kolom(data_mentah, 'Waktu Pesanan Dibuat'))
tanggal_pengganti = tanggal_asli.dropna().median()

data_bersih['tanggal_pesanan'] = tanggal_asli.fillna(tanggal_pengganti)
data_bersih['tanggal_diisi_dari_median'] = tanggal_asli.isna()

data_bersih['status_pesanan'] = bersihkan_teks(ambil_kolom(data_mentah, 'Status Pesanan'))
data_bersih['alasan_pembatalan'] = bersihkan_teks(ambil_kolom(data_mentah, 'Alasan Pembatalan'), default='')
data_bersih['kategori_produk'] = bersihkan_teks(ambil_kolom(data_mentah, 'product_categories'))
data_bersih['jumlah_kategori_produk'] = ubah_angka(ambil_kolom(data_mentah, 'num_product_categories'))
data_bersih['jumlah_barang'] = ubah_angka(ambil_kolom(data_mentah, 'total_qty'))
data_bersih['berat_total_gram'] = ubah_angka(ambil_kolom(data_mentah, 'total_weight_gr'))
data_bersih['jumlah_return'] = ubah_angka(ambil_kolom(data_mentah, 'total_returned_qty'))
data_bersih['diskon_total'] = ubah_angka(ambil_kolom(data_mentah, 'Total Diskon'))
data_bersih['metode_pembayaran'] = bersihkan_teks(ambil_kolom(data_mentah, 'Metode Pembayaran'))
data_bersih['opsi_pengiriman'] = bersihkan_teks(ambil_kolom(data_mentah, 'Opsi Pengiriman'))
data_bersih['kota_kabupaten'] = bersihkan_teks(ambil_kolom(data_mentah, 'Kota/Kabupaten'))
data_bersih['provinsi'] = bersihkan_teks(ambil_kolom(data_mentah, 'Provinsi'))
data_bersih['ongkir_dibayar_pembeli'] = ubah_angka(ambil_kolom(data_mentah, 'Ongkos Kirim Dibayar oleh Pembeli'))
data_bersih['potongan_ongkir'] = ubah_angka(ambil_kolom(data_mentah, 'Estimasi Potongan Biaya Pengiriman'))
data_bersih['total_pembayaran'] = ubah_angka(ambil_kolom(data_mentah, 'Total Pembayaran'))
data_bersih['perkiraan_ongkir'] = ubah_angka(ambil_kolom(data_mentah, 'Perkiraan Ongkos Kirim'))

data_bersih['tahun'] = data_bersih['tanggal_pesanan'].dt.year
data_bersih['bulan'] = data_bersih['tanggal_pesanan'].dt.month
data_bersih['tahun_bulan'] = data_bersih['tanggal_pesanan'].dt.to_period('M').astype(str)
data_bersih['kategori_umum'] = 'Belanja'

data_bersih = data_bersih.dropna(subset=['order_id', 'tanggal_pesanan', 'total_pembayaran']).copy()
data_bersih = data_bersih[data_bersih['status_pesanan'].str.lower().ne('batal')].copy()
data_bersih = data_bersih[data_bersih['total_pembayaran'] > 0].copy()

baris_sebelum_duplikat = len(data_bersih)
data_bersih = data_bersih.drop_duplicates(subset=['order_id']).copy()
duplikat_dihapus = baris_sebelum_duplikat - len(data_bersih)

q1, q3, iqr, batas_bawah, batas_atas = hitung_iqr(data_bersih['total_pembayaran'])

data_bersih['outlier_total_pembayaran'] = (
    (data_bersih['total_pembayaran'] < batas_bawah) |
    (data_bersih['total_pembayaran'] > batas_atas)
)

data_bersih['total_pembayaran_capped'] = data_bersih['total_pembayaran'].clip(
    lower=max(batas_bawah, 0),
    upper=batas_atas
)

data_bersih = data_bersih.sort_values('tanggal_pesanan').reset_index(drop=True)

display(data_bersih.head())

print('Baris raw:', len(data_mentah))
print('Baris clean:', len(data_bersih))
print('Duplikat order_id dihapus:', duplikat_dihapus)
print('Tanggal missing diisi median:', int(data_bersih['tanggal_diisi_dari_median'].sum()))
print('Outlier ditandai:', int(data_bersih['outlier_total_pembayaran'].sum()))
print('Batas atas IQR:', round(batas_atas, 2))

,order_id,dataset_id,nama_dataset,periode,tanggal_pesanan,tanggal_diisi_dari_median,status_pesanan,alasan_pembatalan,kategori_produk,jumlah_kategori_produk,jumlah_barang,berat_total_gram,jumlah_return,diskon_total,metode_pembayaran,opsi_pengiriman,kota_kabupaten,provinsi,ongkir_dibayar_pembeli,potongan_ongkir,total_pembayaran,perkiraan_ongkir,tahun,bulan,tahun_bulan,kategori_umum,outlier_total_pembayaran,total_pembayaran_capped
0,ORD_0006556,ecommerce_2024_01_january,January 2024,2024-01,2024-01-01 00:05:00,False,Selesai,,Aksesoris ID Card,1.0,2.0,50.0,0.0,0.0,Saldo ShopeePay,Hemat Kargo-SPX Hemat,KAB. BEKASI,JAWA BARAT,0.0,8000.0,10000.0,8000.0,2024,1,2024-01,Belanja,False,10000.0
1,ORD_0006557,ecommerce_2024_01_january,January 2024,2024-01,2024-01-01 00:07:00,False,Selesai,,Celengan,1.0,2.0,1000.0,0.0,0.0,COD (Bayar di Tempat),Hemat Kargo-SPX Hemat,KOTA BANDUNG,JAWA BARAT,0.0,10000.0,35663.0,10000.0,2024,1,2024-01,Belanja,False,35663.0
2,ORD_0006558,ecommerce_2024_01_january,January 2024,2024-01,2024-01-01 00:07:00,False,Selesai,,Plastik / Wadah Plastik,1.0,1.0,600.0,0.0,0.0,COD (Bayar di Tempat),Reguler (Cashless)-SPX Standard,KAB. BOGOR,JAWA BARAT,0.0,10000.0,23187.0,10000.0,2024,1,2024-01,Belanja,False,23187.0
3,ORD_0006559,ecommerce_2024_01_january,January 2024,2024-01,2024-01-01 00:12:00,False,Selesai,,Rak / Rak Serbaguna,1.0,1.0,700.0,0.0,0.0,COD (Bayar di Tempat),Hemat Kargo-SPX Hemat,KOTA DEPOK,JAWA BARAT,0.0,8000.0,37400.0,8000.0,2024,1,2024-01,Belanja,False,37400.0
4,ORD_0006560,ecommerce_2024_01_january,January 2024,2024-01,2024-01-01 00:36:00,False,Selesai,,Celengan,1.0,1.0,500.0,0.0,0.0,Online Payment,Hemat Kargo-SPX Hemat,KAB. BEKASI,JAWA BARAT,0.0,8000.0,21800.0,8000.0,2024,1,2024-01,Belanja,False,21800.0


Baris raw: 5196
Baris clean: 4545
Duplikat order_id dihapus: 0
Tanggal missing diisi median: 1717
Outlier ditandai: 491
Batas atas IQR: 88620.0


## 10. Validasi hasil cleaning

Validasi singkat:
- Tidak ada `order_id`, tanggal, amount, status, kategori, kota, provinsi kosong.
- Tidak ada amount 0/negatif.
- Tidak ada `order_id` duplikat.


In [25]:
kolom_penting = [
    'order_id', 'tanggal_pesanan', 'total_pembayaran', 'status_pesanan',
    'kategori_produk', 'metode_pembayaran', 'kota_kabupaten', 'provinsi'
]

validasi_bersih = pd.DataFrame({
    'kolom': kolom_penting,
    'jumlah_missing': [int(data_bersih[kolom].isna().sum()) for kolom in kolom_penting],
})
validasi_bersih['persen_missing'] = (validasi_bersih['jumlah_missing'] / len(data_bersih) * 100).round(2)

masalah_validasi = {
    'amount_tidak_valid': int((data_bersih['total_pembayaran'] <= 0).sum()),
    'order_id_duplikat': int(data_bersih['order_id'].duplicated().sum()),
    'status_batal_tersisa': int(data_bersih['status_pesanan'].str.lower().eq('batal').sum()),
}

display(validasi_bersih)
print(masalah_validasi)

if validasi_bersih['jumlah_missing'].sum() > 0 or sum(masalah_validasi.values()) > 0:
    raise ValueError('Masih ada masalah pada data bersih.')

print('Validasi cleaning lolos.')


,kolom,jumlah_missing,persen_missing
0,order_id,0,0.0
1,tanggal_pesanan,0,0.0
2,total_pembayaran,0,0.0
3,status_pesanan,0,0.0
4,kategori_produk,0,0.0
5,metode_pembayaran,0,0.0
6,kota_kabupaten,0,0.0
7,provinsi,0,0.0


{'amount_tidak_valid': 0, 'order_id_duplikat': 0, 'status_batal_tersisa': 0}
Validasi cleaning lolos.


## 11. Analisis singkat setelah cleaning

Cek hasil final: baris per bulan, top kategori produk, provinsi, metode pembayaran, dan transaksi outlier terbesar.


In [26]:
ringkasan_bersih = (
    data_bersih.groupby(['dataset_id', 'nama_dataset', 'tahun_bulan'])
    .agg(
        jumlah_order=('order_id', 'count'),
        total_penjualan=('total_pembayaran', 'sum'),
        median_pembayaran=('total_pembayaran', 'median'),
        outlier=('outlier_total_pembayaran', 'sum'),
    )
    .reset_index()
)

top_kategori = data_bersih['kategori_produk'].value_counts().head(10).reset_index(name='jumlah_order')
top_provinsi = data_bersih['provinsi'].value_counts().head(10).reset_index(name='jumlah_order')
top_pembayaran = data_bersih['metode_pembayaran'].value_counts().reset_index(name='jumlah_order')
top_outlier = data_bersih[data_bersih['outlier_total_pembayaran']].sort_values('total_pembayaran', ascending=False).head(10)

display(ringkasan_bersih)
display(top_kategori)
display(top_provinsi)
display(top_pembayaran)
display(top_outlier[['order_id', 'tanggal_pesanan', 'kategori_produk', 'provinsi', 'total_pembayaran', 'total_pembayaran_capped']])


,dataset_id,nama_dataset,tahun_bulan,jumlah_order,total_penjualan,median_pembayaran,outlier
0,ecommerce_2024_01_january,January 2024,2024-01,367,24492481.0,35400.0,53
1,ecommerce_2024_06_june,June 2024,2024-06,608,32465925.0,25248.0,67
2,ecommerce_2024_12_december,December 2024,2025-02,1042,56060258.0,22520.0,90
3,ecommerce_2025_02_february,February 2025,2025-02,831,53088026.0,22900.0,117
4,ecommerce_2025_07_july,July 2025,2025-02,675,41642561.0,20500.0,81
5,ecommerce_2025_11_november,November 2025,2025-11,1022,45388311.0,19500.0,83


,kategori_produk,jumlah_order
0,Celengan,1155
1,Mangkok Sambal / Saus,1049
2,Aksesoris Pintu,684
3,Nampan / Tray,251
4,Seal / Baut / Roof,173
5,Lunch Box / Rantang,98
6,Other,97
7,Tempat Nasi,89
8,Keranjang,87
9,Rak / Rak Serbaguna,87


,provinsi,jumlah_order
0,JAWA BARAT,1424
1,BANTEN,772
2,DKI JAKARTA,605
3,JAWA TENGAH,364
4,JAWA TIMUR,359
5,SUMATERA SELATAN,150
6,LAMPUNG,98
7,RIAU,86
8,JAMBI,79
9,SUMATERA UTARA,68


,metode_pembayaran,jumlah_order
0,COD (Bayar di Tempat),2529
1,Saldo ShopeePay,874
2,Online Payment,634
3,SPayLater,307
4,SeaBank Bayar Instan,137
5,Kartu Kredit/Debit,52
6,Alfamart/Alfamidi/Dan+Dan,6
7,Indomaret/i.Saku,5
8,BCA OneKlik,1


,order_id,tanggal_pesanan,kategori_produk,provinsi,total_pembayaran,total_pembayaran_capped
2037,ORD_0009707,2025-02-12 10:54:00,Celengan,JAWA BARAT,3403591.0,88620.0
1180,ORD_0005831,2025-02-05 09:59:00,Seal / Baut / Roof,JAWA BARAT,3023425.0,88620.0
2620,ORD_0004294,2025-02-12 10:54:00,Seal / Baut / Roof,JAWA BARAT,2692454.0,88620.0
1955,ORD_0009716,2025-02-12 10:54:00,Seal / Baut / Roof,JAWA TIMUR,2652000.0,88620.0
1315,ORD_0005985,2025-02-09 23:28:00,Seal / Baut / Roof,DKI JAKARTA,1538033.0,88620.0
1607,ORD_0004073,2025-02-12 10:54:00,Other,JAWA BARAT,1489000.0,88620.0
1240,ORD_0005896,2025-02-07 16:01:00,Seal / Baut / Roof,JAWA BARAT,1441900.0,88620.0
3653,ORD_0015184,2025-11-04 16:55:00,Lunch Box / Rantang,BANTEN,1415660.0,88620.0
1525,ORD_0009582,2025-02-12 10:54:00,Seal / Baut / Roof,JAWA BARAT,1370000.0,88620.0
3985,ORD_0015555,2025-11-15 09:57:00,Seal / Baut / Roof,DKI JAKARTA,1360900.0,88620.0


## 13. Simpan 1 dataset clean

Hanya satu file disimpan ke `data/clean`: `ecommerce_sales_clean.csv`.

Kolom sudah rapi dan siap analisis. Nilai original `total_pembayaran` tetap ada. Kolom `total_pembayaran_capped` disediakan jika analisis butuh versi outlier yang sudah dibatasi IQR.


In [27]:
file_output = folder_clean / 'ecommerce_sales_clean.csv'

data_bersih.to_csv(file_output, index=False)

print('File clean tersimpan:', file_output)
print('Jumlah baris:', len(data_bersih))
print('Jumlah kolom:', data_bersih.shape[1])


File clean tersimpan: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/clean/ecommerce_sales_clean.csv
Jumlah baris: 4545
Jumlah kolom: 28


## 14. Variabel penting

- `data_mentah`: gabungan 6 file raw e-commerce.
- `data_bersih`: satu dataset e-commerce yang sudah rapi.
- `ringkasan_raw`: ringkasan file mentah.
- `ringkasan_bersih`: ringkasan hasil cleaning per bulan.
- `validasi_bersih`: validasi kolom penting.
